In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as an
import scipy
import sklearn
import gget

sc.settings.verbosity = 3  

In [3]:
# load the single-cell data
fpath = "/scratch/indikar_root/indikar1/shared_data/single_cell_fibroblast/scanpy/processed.anndata.h5ad"
adata = sc.read_h5ad(fpath)
sc.logging.print_memory_usage()
adata

Memory usage: current 4.72 GB, difference +2.05 GB


AnnData object with n_obs × n_vars = 7748 × 14082
    obs: 'n_genes', 'cell', 'G1', 'G2M', 'S', 'pred_phase', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'leiden', 'pred_G1', 'pred_S', 'pred_G2M', 'dpt_pseudotime'
    var: 'gene_name', 'Chromosome', 'Start', 'End', 'Strand', 'seurat_S', 'seurat_G2M', 'is_seurat', 'is_kegg', 'whitfield_G1/S', 'whitfield_G2', 'whitfield_G2/M', 'whitfield_M/G1', 'whitfield_S', 'is_whitfield', 'GO_G1', 'GO_G1/S', 'GO_G2', 'GO_G2/M', 'GO_M', 'GO_S', 'is_GO', 'LIU_G2MvsG0G1', 'LIU_G2MvsS', 'LIU_SvsG0G1', 'is_LIU', 'cell_cycle', 'n_cells', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_count

In [21]:
# load the population data
fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/data/lab_data/rajapakse_lab_data.h5ad"
bdata = sc.read_h5ad(fpath)
bdata = bdata[bdata.obs['dataset'] == 'chen_2015', :].copy()
sc.logging.print_memory_usage()
bdata

Memory usage: current 4.81 GB, difference +0.04 GB


AnnData object with n_obs × n_vars = 18 × 19393
    obs: 'dataset', 'sample_id', 'timepoint', 'hour', 'n_counts', 'control'
    var: 'gene_id', 'token_id', 'Chromosome', 'Source', 'Feature', 'Start', 'End', 'Score', 'Strand', 'Frame', 'gene_version', 'gene_source', 'gene_biotype', 'transcript_id', 'transcript_version', 'transcript_name', 'transcript_source', 'transcript_biotype', 'tag', 'ccds_id', 'exon_number', 'exon_id', 'exon_version', 'protein_id', 'protein_version', 'transcript_support_level', 'ensembl_id'

In [22]:
bdata.var.head()

,gene_id,token_id,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,...,transcript_biotype,tag,ccds_id,exon_number,exon_id,exon_version,protein_id,protein_version,transcript_support_level,ensembl_id
gene_name,,,,,,,,,,,,,,,,,,,,,
A1BG,ENSG00000121410,5150.0,19,ensembl_havana,gene,58345177.0,58353492.0,.,-,.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSG00000121410
A1CF,ENSG00000148584,9064.0,10,ensembl_havana,gene,50799408.0,50885675.0,.,-,.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSG00000148584
A2M,ENSG00000175899,13826.0,12,ensembl_havana,gene,9067663.0,9116229.0,.,-,.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSG00000175899
A2ML1,ENSG00000166535,11812.0,12,ensembl_havana,gene,8822620.0,8887001.0,.,+,.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSG00000166535
A3GALT2,ENSG00000184389,15327.0,1,ensembl_havana,gene,33306765.0,33321098.0,.,-,.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ENSG00000184389


In [35]:
threshold = 5
pdf = bdata.to_df()
gdf = pdf.sum(axis=0).sort_values(ascending=False)
gdf = gdf.reset_index()
gdf.columns = ['gene_name', 'sum']
gdf['is_expressed'] = gdf['sum'] > threshold
gdf['is_expressed'].value_counts()

is_expressed
True     14436
False     4957
Name: count, dtype: int64

In [37]:
# what is the overlap?
population_genes = gdf[gdf['is_expressed']]['gene_name'].to_list()
single_cell_genes = list(adata.var_names)

print(f"{len(population_genes)=}")
print(f"{len(single_cell_genes)=}")

len(population_genes)=14436
len(single_cell_genes)=14082


In [38]:
def compare_lists(list1, list2):
  set1 = set(list1)
  set2 = set(list2)
  in_common = set1 & set2
  in_list1_not_list2 = set1 - set2
  in_list2_not_list1 = set2 - set1
  return in_common, in_list1_not_list2, in_list2_not_list1

common, in_population, in_single_cell = compare_lists(population_genes, single_cell_genes)
print(f"Number of elements in common: {len(common)}")
print(f"Number of elements in population_genes but not sc_genes: {len(in_population)}")
print(f"Number of elements in sc_genes but not population_genes: {len(in_single_cell)}")

Number of elements in common: 10488
Number of elements in population_genes but not sc_genes: 3948
Number of elements in sc_genes but not population_genes: 3594


In [40]:
list(sorted(in_population))[:10]

['A4GALT',
 'AACS',
 'AANAT',
 'AATF',
 'ABCA13',
 'ABCA2',
 'ABCA3',
 'ABCA7',
 'ABCA9',
 'ABCB9']

In [41]:
list(sorted(in_single_cell))[:10]

['A2ML1',
 'A3GALT2',
 'A4GNT',
 'AADAC',
 'AADACL2',
 'AADACL3',
 'AADACL4',
 'AADAT',
 'ABCB11',
 'ABCF2-H2BK1']

In [49]:
sdf = adata.to_df(layer='counts')
sdf = sdf[list(in_single_cell)]

sdf.sum(axis=0).sort_values(ascending=False)

CEACAM3     1143791
FOXL2NB      679957
KIR2DL3      550926
WNT3A        491525
SLC1A2       487657
             ...   
PITX1             5
C1orf226          5
GSTT2B            5
CIBAR2            5
CLDN24            5
Length: 3594, dtype: int64

In [51]:
[x for x in bdata.var_names if 'CEACAM3' in x]

['CEACAM3']